# Reinforcement Learning — Implementations from David Silver's UCL Course (Lectures 1–5)

My own from-scratch implementations of the core algorithms from the first five lectures of David
Silver's [Reinforcement Learning course](https://www.davidsilver.uk/teaching/) (UCL), interleaved with
my own notes on the theory. Everything below — environments, agents, and algorithms — is implemented
directly with NumPy (and a `gymnasium.Env` interface for the grid-world environments), not called from
an existing RL library.

**Contents**
- Lectures 1–2: MDPs, policies, value functions, Bellman equations (theory)
- Lecture 3: Planning by Dynamic Programming — Policy Iteration & Value Iteration (Gridworld)
- Lecture 4: Model-Free Prediction — Monte Carlo & TD(λ) (Random Walk)
- Lecture 5: Model-Free Control — MC Control, SARSA, SARSA(λ), Q-Learning (Windy Gridworld & Cliff Walking)


## Lectures 1–2: Markov Decision Processes

Core definitions and the Bellman equations, following the course notation.

---

### policy:

$$\pi(a \mid s) = P(A_t = a \mid S_t = s)$$

### Types of Policies

### 1. Deterministic Policy

$$\pi(s) = a$$

### 2. Stochastic Policy

$$
\pi(a \mid s) \in [0,1], \quad \sum_a \pi(a \mid s) = 1
$$

### Optimal Policy

$$
\pi^* = \arg\max_\pi \mathbb{E}[R]
$$

---
### Reward:
$$
R^a_s = \mathbb{E}[R_{t+1} | S_t=s,A_t=a]
$$

---
### Value Function:

$$
G_t = \sum_{k=0}^{\infty} \gamma^k \, R_{t+k+1} (return)
$$
$$
V\pi(s) = \mathbb{E}_\pi \left[ R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots \mid S_t = s \right]
$$
$$
V\pi(s) = \mathbb{E}_\pi \left[G_t \mid S_t = s \right]
$$

---
### action_value Function:
$$
q(s,a) = \mathbb{E}_\pi[G_t \mid S_t=s , A_t=a ]
$$

---
### transition probabilities:
$$
P^a_{ss'} = P[S_{t+1} \mid S_t=s , A_t=a ]
$$

---
Markov : independent of the past (present containing all the information the past)

$$
P[S_{t+1} \mid S_t] = P[S_{t+1} \mid S_t , S_{t-1} , ... , S_1]
$$

---

## Exploration vs Exploitation

Policies must balance:

- **Exploration**: trying new actions
- **Exploitation**: choosing the best-known action

Example strategies:
- ε-greedy policy
- Softmax policy

----

**Prediction** : evaluate the future , Given th policy

**Control** : optimize the future , Finds the best policy

---
### Bellman eq:

$$
V(s) = \mathbb{E}_\pi[ R_{t+1} + \gamma V(s+1) \mid S_t=s ]
$$

$$ V(s) = R_s + \gamma \sum_{s' \in S} P_{ss'} V(s') $$

in code: $v = R + \gamma Pv$ (vectors)

$$
q(s,a) = \mathbb{E}_\pi[R_{t+1} + \gamma q_{\pi} (s_{t+1} , a_{t+1}) \mid S_t= s ,A_t=a]
$$

$$ q(s,a) = R^a_s + \gamma \sum_{s' \in S} P_{ss'} v(s') $$

in code: $q(s,a)

$$ \Rightarrow V_\pi = \sum_{a \in A} \pi(a \mid s) q_\pi (s,a)$$

### the optimals:
$$V_*(s) = \max V_{\pi}(s)$$
$$q_*(s,a) = \max q_{\pi}(s,a)$$

$$ \Rightarrow V_* = \max_a q_*(s,a) $$

---
## MRP (Markov Reward Process) ⇒ $<S,P,R,\gamma>$
## MDP (Markov Decision Process) ⇒ $<S ,A,P,R,\gamma>$
## POMDP (Partially Observed MDP) ⇒ $<S,A,O,P,R,Z, \gamma>$   
(O $\to$ infinite set of observations , Z $\to$ observation function)

## Lecture 3: Planning by Dynamic Programming

Solving a known MDP (Gridworld) exactly via Value Iteration and Policy Iteration.

---
### Policy Iteration:
***Policy evaluation*** Estimate $V_\pi$, Iterative policy evaluation

***Policy improvement*** Generate $\pi' \geq \pi $,  Greedy policy improvement

(for Generalised Policy Iteration we can use **Any** policy evaluation algorithm and **Any** policy improvement algorithm)

example:

In [1]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


In [2]:
class GridworldEnv(gym.Env):
    """
    Gridworld environment with walls, multiple terminal states,
    and random non-blocking spawns.
    """
    def __init__(self, grid_size=5):
        super().__init__()
        self.grid_size = grid_size

        self.observation_space = spaces.Tuple((
            spaces.Discrete(self.grid_size),
            spaces.Discrete(self.grid_size)
        ))
        self.action_space = spaces.Discrete(4)

        self.terminals = {(0, 0): 10.0, (4, 4): -1.0}
        self.walls = {(1, 2), (2, 2), (3, 1)}

        self.action_effects = {
            0: (-1, 0),  # Up
            1: (0, 1),   # Right
            2: (1, 0),   # Down
            3: (0, -1)   # Left
        }
        self.state = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        while True:
            row = self.np_random.integers(0, self.grid_size)
            col = self.np_random.integers(0, self.grid_size)
            state = (row, col)

            if state not in self.terminals and state not in self.walls:
                self.state = state
                break

        return self.state, {}

    def step(self, action):
        if self.state in self.terminals:
            return self.state, 0.0, True, False, {}

        dr, dc = self.action_effects[action]
        next_row = int(np.clip(self.state[0] + dr, 0, self.grid_size - 1))
        next_col = int(np.clip(self.state[1] + dc, 0, self.grid_size - 1))
        next_state = (next_row, next_col)

        if next_state not in self.walls:
            self.state = next_state

        terminated = self.state in self.terminals
        reward = self.terminals[self.state] if terminated else -1.0

        return self.state, reward, terminated, False, {}

    def render(self):
        if self.state is None:
            return

        grid = np.full((self.grid_size, self.grid_size), ".", dtype=object)

        for r, c in self.walls:
            grid[r, c] = "#"
        for (r, c), reward in self.terminals.items():
            grid[r, c] = "G" if reward > 0 else "B"

        grid[self.state[0], self.state[1]] = "A"

        print("\n".join(" ".join(row) for row in grid) + "\n")

In [3]:
gridworld_env = GridworldEnv()
gridworld_env.reset()
gridworld_env.render()

G . . A .
. . # . .
. . # . .
. # . . .
. . . . B



In [4]:
class ValueIterationAgent:
    def __init__(self, env, discount=0.9):
        self.env = env
        self.discount = discount
        self.V = np.zeros((env.grid_size, env.grid_size))

        # Pre-populate terminal values in V matrix
        for state, reward in self.env.terminals.items():
            self.V[state] = reward

    def run(self, iterations=1000, theta=1e-4):
        actions = list(self.env.action_effects.values())
        size = self.env.grid_size

        for _ in range(iterations):
            delta = 0
            V_new = np.copy(self.V)

            for r in range(size):
                for c in range(size):
                    state = (r, c)

                    if state in self.env.terminals or state in self.env.walls:
                        continue

                    action_values = []
                    for dr, dc in actions:
                        nr = int(np.clip(r + dr, 0, size - 1))
                        nc = int(np.clip(c + dc, 0, size - 1))
                        next_state = state if (nr, nc) in self.env.walls else (nr, nc)

                        reward = self.env.terminals[next_state] if next_state in self.env.terminals else -1.0
                        value = reward + self.discount * self.V[next_state]
                        action_values.append(value)

                    max_value = max(action_values)
                    delta = max(delta, abs(max_value - self.V[r, c]))
                    V_new[r, c] = max_value

            self.V = V_new
            if delta < theta:
                break

    def display(self):
        print("\n===== VALUE FUNCTION =====")
        for r in range(self.env.grid_size):
            row_str = []
            for c in range(self.env.grid_size):
                state = (r, c)
                if state in self.env.walls:
                    row_str.append("  WALL ")
                else:
                    row_str.append(f"{self.V[r, c]:7.2f}")
            print(" ".join(row_str))

In [5]:
vi_agent = ValueIterationAgent(gridworld_env)
vi_agent.run()
vi_agent.display()


===== VALUE FUNCTION =====
  10.00   19.00   16.10   13.49   11.14
  19.00   16.10   WALL    11.14    9.03
  16.10   13.49   WALL     9.03    7.12
  13.49   WALL     5.41    7.12    5.41
  11.14    9.03    7.12    5.41   -1.00


In [6]:
class PolicyIterationAgent:
    def __init__(self, env, discount=0.9):
        self.env = env
        self.discount = discount
        self.size = env.grid_size

        self.V = np.zeros((self.size, self.size))
        self.policy = np.random.randint(0, 4, (self.size, self.size))

        # Pre-populate terminal state rewards into the value matrix
        for state, reward in self.env.terminals.items():
            self.V[state] = reward

    def policy_evaluation(self, iterations=500, theta=1e-6):
      #this part is the same with value iteration class
        for _ in range(iterations):
            delta = 0.0
            V_new = np.copy(self.V)

            for r in range(self.size):
                for c in range(self.size):
                    state = (r, c)

                    if state in self.env.terminals or state in self.env.walls:
                        continue

                    action = self.policy[r, c]
                    dr, dc = self.env.action_effects[action]

                    nr = int(np.clip(r + dr, 0, self.size - 1))
                    nc = int(np.clip(c + dc, 0, self.size - 1))
                    next_state = state if (nr, nc) in self.env.walls else (nr, nc)

                    reward = self.env.terminals[next_state] if next_state in self.env.terminals else -1.0
                    new_state_value = reward + self.discount * self.V[next_state]

                    delta = max(delta, abs(new_state_value - self.V[r, c]))
                    V_new[r, c] = new_state_value

            self.V = V_new
            if delta < theta:
                break

    def policy_improvement(self):
        stable = True

        for r in range(self.size):
            for c in range(self.size):
                state = (r, c)

                if state in self.env.terminals or state in self.env.walls:
                    continue

                old_action = self.policy[r, c]
                values = []

                for action in range(4):
                    dr, dc = self.env.action_effects[action]

                    nr = int(np.clip(r + dr, 0, self.size - 1))
                    nc = int(np.clip(c + dc, 0, self.size - 1))
                    next_state = state if (nr, nc) in self.env.walls else (nr, nc)

                    reward = self.env.terminals[next_state] if next_state in self.env.terminals else -1.0
                    values.append(reward + self.discount * self.V[next_state])

                best_action = np.argmax(values)
                self.policy[r, c] = best_action

                if old_action != best_action:
                    stable = False

        return stable

    def run(self, max_iterations=1000, theta=1e-6):
        for _ in range(max_iterations):
            self.policy_evaluation(theta=theta)
            if self.policy_improvement():
                break

    def display(self):
        symbols = ["↑", "→", "↓", "←"]

        print("\n===== POLICY =====")
        for r in range(self.size):
            row_str = []
            for c in range(self.size):
                state = (r, c)
                if state in self.env.walls:
                    row_str.append("#####")
                elif state in self.env.terminals:
                    row_str.append("  T  ")
                else:
                    row_str.append(f"  {symbols[self.policy[r, c]]}  ")
            print(" | ".join(row_str))

In [7]:
pi_agent = PolicyIterationAgent(gridworld_env)
pi_agent.run()
pi_agent.display()


===== POLICY =====
  T   |   ←   |   ←   |   ←   |   ←  
  ↑   |   ↑   | ##### |   ↑   |   ↑  
  ↑   |   ↑   | ##### |   ↑   |   ↑  
  ↑   | ##### |   →   |   ↑   |   ↑  
  ↑   |   ←   |   ←   |   ↑   |   T  


---
**Input:** MDP $M = \langle S, s_0, A, P_a(s' \mid s), r(s,a,s') \rangle$  
**Output:** Value function $V$

Set $V$ to arbitrary value function; e.g., $V(s) = 0$ for all $s$

---

**repeat**:

>$\Delta \leftarrow 0$

>for each $s \in S$:

>>$
V'(s) \leftarrow \max_{a \in A(s)} \sum_{s' \in S} P_a(s' \mid s)\,\big[ r(s,a,s') + \gamma V(s') \big]
$

>>$
\Delta \leftarrow \max \left( \Delta, \left| V'(s) - V(s) \right| \right)
$

>$
V \leftarrow V'
$

**until** $\Delta \leq \theta$

---
also we can use ***action_value function***:

$
\Delta \leftarrow 0
$

**for each** $s \in S$:

>**for each** $a \in A(s)$

>>$
\quad Q(s,a) \leftarrow \sum_{s' \in S} P_a(s' \mid s)\,\big[ r(s,a,s') + \gamma V(s') \big]
$

>$
\Delta \leftarrow \max \left( \Delta,\; \left| \max_{a \in A(s)} Q(s,a) - V(s) \right| \right)
$

>$
V(s) \leftarrow \max_{a \in A(s)} Q(s,a)
$

---

## Lecture 4: Model-Free Prediction

Estimating $V_\pi$ from experience, without knowing the MDP's transition/reward model, on a Random Walk chain: Monte Carlo prediction, then TD(0) and TD(λ).

## First_Visit Monte Carlo:
by the law of large numbers it will converge to $V_{\pi}$ by N → $\infty$

Initialize:
>π = policy to be evaluated

>V = arbitrary state-value function (e.g., initialized to 0)

>Returns(s) = an empty list for all s ∈ S

Loop forever (for each episode):

>1. Generate an episode following $\pi: S_0, A_0, R_1, S_1, A_1, R_2, ..., S_{T-1}, A_{T-1}, R_T$
>2. G ← 0
>3. Loop for each step of the episode,$ t = T-1, T-2, ..., 0$:

>>a. $G ← γG + R_{t+1}  $       

>>b. If S_t does not appear in $S_0, S_1, ..., S_{t-1}$:

>>>i.  Append G to Returns$(S_t)$

>>>ii. $V(S_t) ← average(Returns(S_t))$

---
Every-Visit Monte Carlo:

Initialize:
>π = policy to be evaluated

>V = arbitrary state-value function (e.g., initialized to 0)

>Returns(s) = an empty list for all s ∈ S

Loop forever (for each episode):
>1. Generate an episode following $\pi: S_0, A_0, R_1, S_1, A_1, R_2, ..., S_{T-1}, A_{T-1}, R_T$
>2. G ← 0
>3. Loop for each step of the episode, t = T-1, T-2, ..., 0:

>>a. G ← γG + $R_{t+1}$

>>b. Append G to Returns$(S_t)$

>>c. $V(S_t) ← average(Returns(S_t))$

---
Now if we want to compare them:

In **FVMC** we just update the V(s) when we first visit it but in **EVMC** we update the V(s) on every visit.
Also **FVMC** is *Unbiased estimator* and *Typically lower variance early on*,
But **EVMC** is *Biased in the behavior initially, but collapses asymptotically to unbiased* and *Typically higher variance initially, but utilizes more data per episode.*

---
## TD

$$
V(S_t) = V(S_t) + \alpha (G_t - V(S_t) )
$$

for ***TD(0)** : (we only go 1 step) $$ V(S_t) = V(S_t) + \alpha ( R_t + \gamma V(S_{t+1}) - V(S_t)) $$

+ R_t + \gamma V(S_{t+1}) is called *TD target*
+ $\delta_{t} = R_t + \gamma V(S_{t+1}) - V(S_t) $ is *TD error* .

---
### Differences between TD and MC:
+ **Return $G_t$** is unbiased , but **TD target** is biased (because it is a estimation) .  
+ **TD target** is much lower variance than **the Return**:
+ Return depends on many random actions, transitions, rewards
+ TD target depends on one random action, transition, reward

+ TD exploits Markov property ⇒ Usually more efficient in Markov environments
+ MC does not exploit Markov property ⇒ Usually more effective in non-Markov environments



#### MC has high variance, zero bias
+ Good convergence properties
+ (even with function approximation)
+ Not very sensitive to initial value
+ Very simple to understand and use
#### TD has low variance, some bias
+ Usually more efficient than MC
+ TD(0) converges to $v_{\pi}(s)$
+ (but not always with function approximation)
+ More sensitive to initial value

---
#### Review of MLE & MSE:
MSE : measures the average of the squares of the errors

$$ 1/n ( \sum ( Y_i - \hat{Y_i} )^2) $$

MLE :  a method of estimating the parameters of a probability distribution by maximizing a likelihood function, so that under the assumed statistical model the observed data is most probable.

+ MC converges to solution with minimum mean-squared
+ TD(0) converges to solution of max likelihood Markov model

---
## TD($λ$)
in TD($λ$) $G_t$ is different:(TD($λ$) Weighting Function)
$$ G^λ_t = (1 - λ) \sum_{n=0}^{\infty} λ^{n-1} G_t^{(n)}
$$

### Types of TD($λ$):
+ Forward-view TD($λ$) : Like MC, can only be computed from complete episodes , Update value function towards the λ-return
+ Backward View TD($λ$) : Update online, every step, from incomplete sequences
+ Forward view provides theory ,But
Backward view provides mechanism


### We have another concept in TD($λ$) is called **Eligibility Traces**
Frequency heuristic: assign credit to most frequent states

Recency heuristic: assign credit to most recent states

Eligibility traces combine both heuristics
$$
E_0(s) = 0
$$
$$
E_t(s) = γ λ E_{t-1}(s) + 1(S_t = s)
$$

(when λ=0 , E_0(s) = 1(S_t = s) )


 Forward-view TD($λ$):
 $$ V(S_t) ⟵ V(S_t) + α ( G^λ_t - V(S_t)) $$

 Backward View TD($λ$):
 $$ V(s) ⟵  V(s) + α δ_t E_t (s) $$

 + Theorem : The sum of **offline** updates is identical for forward-view and
backward-view TD(λ)

---
***Offline*** Updates are accumulated within episode ,but applied in batch at the end of episode

***Online*** updates:
TD(λ) updates are applied online at each step within episode and Forward and backward-view TD(λ) are slightly different
(NEW: **Exact online** TD(λ) achieves perfect equivalence
By using a slightly different form of eligibility trace)


In [8]:
class RandomWalk:
    def __init__(
        self,
        num_states=5,
        left_reward=0.0,
        right_reward=1.0,
        step_reward=0.0
    ):
        self.num_states = num_states

        self.left_terminal = 0
        self.right_terminal = num_states + 1

        self.left_reward = left_reward
        self.right_reward = right_reward
        self.step_reward = step_reward

        self.current_state = None

    def reset(self):
        self.current_state = np.random.randint(1, self.num_states + 1)
        return self.current_state

    def step(self, action):

        if self.current_state in (self.left_terminal, self.right_terminal):
            return self.current_state, 0.0, True

        if action == 0:
            self.current_state -= 1
        elif action == 1:
            self.current_state += 1
        else:
            raise ValueError("Action must be 0 (left) or 1 (right)")

        # Terminal checks
        if self.current_state == self.left_terminal:
            return self.current_state, self.left_reward, True

        if self.current_state == self.right_terminal:
            return self.current_state, self.right_reward, True

        return self.current_state, self.step_reward, False

    def render(self):
        grid = []

        grid.append("[X]" if self.current_state == self.left_terminal else "[ ]")

        for s in range(1, self.num_states + 1):
            grid.append(" X " if self.current_state == s else " . ")

        grid.append("[X]" if self.current_state == self.right_terminal else "[ ]")

        print("".join(grid))

In [9]:
NUM_STATES = 6
NUM_ACTIONS = 2

randomwalk_env = RandomWalk(num_states=NUM_STATES)
randomwalk_env.reset()
randomwalk_env.render()

[ ] .  .  .  .  X  . [ ]


In [10]:
class MonteCarloPrediction:
    def __init__(self, num_states, num_actions, discount_factor=1.0):
        self.num_states = num_states
        self.num_actions = num_actions
        self.gamma = discount_factor

        self.policy = lambda state: np.random.choice([0, 1])

        # Value table arrays (including terminal state padding)
        self.V = np.zeros(num_states + 2)
        self.Q = np.zeros((num_states + 2, num_actions))

        # Track history of returns for non-terminal states
        self.returns_v = {s: [] for s in range(1, num_states + 1)}
        self.returns_q = {
            (s, a): []
            for s in range(1, num_states + 1)
            for a in range(num_actions)
        }

    def run_episode(self, randomwalk_env):
        """Generates an episode using the internal class policy."""
        episode = []
        state = randomwalk_env.reset()
        done = False

        while not done:
            action = self.policy(state)
            next_state, reward, done = randomwalk_env.step(action)
            episode.append((state, action, reward))
            state = next_state

        return episode

    def mc_evaluation_v(self, randomwalk_env, num_episodes=1000, mode="first-visit"):
        """Evaluates the state-value function V(s)."""
        if mode not in ("first-visit", "every-visit"):
            raise ValueError("Mode must be 'first-visit' or 'every-visit'")

        episode = self.run_episode(randomwalk_env)
        G = 0.0

        # Keep track of what we hit as we walk backwards
        seen_later_in_episode = set()

        for t in reversed(range(len(episode))):
            state, _, reward = episode[t]
            G = reward + self.gamma * G

            # Ignore terminal states if they fall outside tracked bounds
            if state not in self.returns_v:
                continue

            if mode == "first-visit":
                # If seen later in the reversed loop, it means it's a future
                # visit, so this current step 't' is the true first-visit.
                if state in seen_later_in_episode:
                    continue
                seen_later_in_episode.add(state)

            self.returns_v[state].append(G)
            self.V[state] = np.mean(self.returns_v[state])

        return self.V

    def mc_evaluation_q(self, randomwalk_env, num_episodes=1000):
        """Evaluates the state-action value function Q(s,a) using first-visit MC."""

        episode = self.run_episode(randomwalk_env)
        G = 0.0
        seen_later_in_episode = set()

        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = reward + self.gamma * G

            if (state, action) not in self.returns_q:
                continue

            if (state, action) in seen_later_in_episode:
                continue
            seen_later_in_episode.add((state, action))

            self.returns_q[(state, action)].append(G)
            self.Q[state, action] = np.mean(self.returns_q[(state, action)])

        return self.Q

In [11]:
EPISODES = 1000

custom_policy = lambda state: np.random.choice([0, 1])

mc_first = MonteCarloPrediction(NUM_STATES, NUM_ACTIONS)
mc_every = MonteCarloPrediction(NUM_STATES, NUM_ACTIONS)
mc_q     = MonteCarloPrediction(NUM_STATES, NUM_ACTIONS)

for _ in range(EPISODES) :
    first_visit_V = mc_first.mc_evaluation_v(randomwalk_env, mode="first-visit")
    every_visit_V = mc_every.mc_evaluation_v(randomwalk_env, mode="every-visit")
    mc_Q = mc_q.mc_evaluation_q(randomwalk_env)

print("First-Visit V(s):")
print(np.round(first_visit_V[1:-1], 3))

print("\nEvery-Visit V(s):")
print(np.round(every_visit_V[1:-1], 3))

print("\nFirst-Visit Q(s,a):")
print(np.round(mc_Q[1:-1], 3))

First-Visit V(s):
[0.159 0.281 0.43  0.57  0.693 0.836]

Every-Visit V(s):
[0.149 0.293 0.419 0.563 0.704 0.836]

First-Visit Q(s,a):
[[0.    0.308]
 [0.154 0.397]
 [0.259 0.544]
 [0.397 0.704]
 [0.571 0.87 ]
 [0.722 1.   ]]


In [12]:
class TDAgents:
    def __init__(self, num_states, num_actions, alpha=0.1, gamma=1.0, lambd=0.5):
        """Initializes Temporal Difference evaluation agents for state values V(s)."""
        self.num_states = num_states
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.lambd = lambd

        # State values V(s) and Eligibility traces E(s)
        self.V = np.zeros(num_states + 2)
        self.E = np.zeros(num_states + 2)

    def train_episode_td0(self, randomwalk_env, policy, max_steps=2000):
        """Trains state values using standard online TD(0) evaluation."""
        state = randomwalk_env.reset()
        done = False
        steps = 0

        while not done and steps < max_steps:
            action = policy(state)
            next_state, reward, done = randomwalk_env.step(action)
            steps += 1

            td_error = reward + self.gamma * self.V[next_state] - self.V[state]
            self.V[state] += self.alpha * td_error
            state = next_state

        return self.V

    def train_episode_td_lambda(self, randomwalk_env, policy, max_steps=2000):
        """Trains state values using standard online TD(lambda) backward view."""
        self.E.fill(0.0)
        state = randomwalk_env.reset()
        done = False
        steps = 0

        while not done and steps < max_steps:
            action = policy(state)
            next_state, reward, done = randomwalk_env.step(action)
            steps += 1

            self.E[state] += 1.0
            td_error = reward + self.gamma * self.V[next_state] - self.V[state]

            self.V += self.alpha * td_error * self.E
            self.E *= self.gamma * self.lambd
            state = next_state

        return self.V

    def train_episode_forward_offline(self, randomwalk_env, policy, max_steps=1000):
        """Trains state values using offline episodic TD(lambda) forward view."""
        trajectory = []
        state = randomwalk_env.reset()
        done = False
        steps = 0

        # Step 1: Collect trajectory history bounds
        while not done and steps < max_steps:
            action = policy(state)
            next_state, reward, done = randomwalk_env.step(action)
            steps += 1

            trajectory.append((state, reward, next_state))
            state = next_state

        # Step 2: Epoched matrix updates
        T = len(trajectory)
        V_old = np.copy(self.V)
        updates = np.zeros_like(self.V)

        for t in range(T):
            state_t = trajectory[t][0]
            lambda_return = 0.0

            for n in range(1, T - t + 1):
                g_n = sum((self.gamma ** k) * trajectory[t + k][1] for k in range(n))

                if t + n < T:
                    g_n += (self.gamma ** n) * V_old[trajectory[t + n][0]]
                    weight = (1.0 - self.lambd) * (self.lambd ** (n - 1))
                else:
                    g_n += (self.gamma ** n) * V_old[trajectory[-1][2]]
                    weight = self.lambd ** (n - 1)

                lambda_return += weight * g_n

            updates[state_t] += self.alpha * (lambda_return - V_old[state_t])

        self.V += updates
        return self.V

    def train_episode_backward_offline(self, randomwalk_env, policy, max_steps=2000):
        """Trains state values using offline episodic TD(lambda) backward view."""
        self.E.fill(0.0)
        updates = np.zeros_like(self.V)
        V_old = np.copy(self.V)

        state = randomwalk_env.reset()
        done = False
        steps = 0

        while not done and steps < max_steps:
            action = policy(state)
            next_state, reward, done = randomwalk_env.step(action)
            steps += 1

            self.E[state] += 1.0
            td_error = reward + self.gamma * V_old[next_state] - V_old[state]

            updates += self.alpha * td_error * self.E
            self.E *= self.gamma * self.lambd
            state = next_state

        self.V += updates
        return self.V

    def train_episode_forward_online(self, randomwalk_env, policy, max_steps=1000):
        """Trains state values using online step-by-step TD(lambda) forward view."""
        trajectory = []
        state = randomwalk_env.reset()
        done = False
        steps = 0

        while not done and steps < max_steps:
            action = policy(state)
            next_state, reward, done = randomwalk_env.step(action)
            steps += 1

            trajectory.append((state, reward, next_state))
            t_current = len(trajectory) - 1

            for t in range(t_current + 1):
                state_t = trajectory[t][0]
                n = t_current - t + 1

                g_n = sum((self.gamma ** k) * trajectory[t + k][1] for k in range(n))
                g_n += (self.gamma ** n) * self.V[next_state]

                weight = (1.0 - self.lambd) * (self.lambd ** (n - 1)) if not done else (self.lambd ** (n - 1))
                self.V[state_t] += self.alpha * weight * (g_n - self.V[state_t])

            state = next_state

        return self.V


In [13]:
ALPHA = 0.05
GAMMA = 1.0
LAMBDA = 0.4
EPISODES = 1000

def random_policy(state):
    return np.random.choice([0, 1])

training_methods = {
    "TD(0)": TDAgents.train_episode_td0,
    "TD(λ) - Forward Offline": TDAgents.train_episode_forward_offline,
    "TD(λ) - Backward Offline": TDAgents.train_episode_backward_offline,
    "TD(λ) - Forward Online": TDAgents.train_episode_forward_online,
    "TD(λ) - Standard Backward Online": TDAgents.train_episode_td_lambda
}

for name, method in training_methods.items():
    agent = TDAgents( num_states=NUM_STATES, num_actions=NUM_ACTIONS, alpha=ALPHA, gamma=GAMMA, lambd=LAMBDA)

    for _ in range(EPISODES):
        method(agent, randomwalk_env, random_policy)

    estimated_values = agent.V[1:-1]

    print(f"Method: {name}: {np.round(estimated_values, 3)}")
    print("-" * 75)

Method: TD(0): [0.115 0.263 0.417 0.577 0.736 0.891]
---------------------------------------------------------------------------


Method: TD(λ) - Forward Offline: [0.188 0.397 0.504 0.651 0.782 0.889]
---------------------------------------------------------------------------


Method: TD(λ) - Backward Offline: [0.112 0.272 0.398 0.533 0.668 0.872]
---------------------------------------------------------------------------


Method: TD(λ) - Forward Online: [0.131 0.212 0.372 0.538 0.701 0.896]
---------------------------------------------------------------------------
Method: TD(λ) - Standard Backward Online: [0.098 0.239 0.421 0.588 0.717 0.85 ]
---------------------------------------------------------------------------


## Lecture 5: Model-Free Control

Learning a good policy from experience: on-policy Monte Carlo control (GLIE), SARSA / SARSA(λ) on Windy Gridworld, and off-policy Q-Learning, including the classic SARSA-vs-Q-Learning comparison on Cliff Walking.

### On-policy learning
+ Learn on the job"
+ Learn about policy π from experience sampled from π
### Off-policy learning
+ Look over someone's shoulder"
+ Learn about policy π from experience sampled from π

---
## Greedy improvement
### over V(S)
$$ \pi'(s) = \arg \max_{a \in A} R^a_s + P^a_{ss'} V(s')  $$
### over Q(s,a)
$$ \pi'(s) = \arg \max_{a \in A} Q(s,a) $$


#### for maintaining the exploration:
$$
\pi(a|s) = \begin{cases} \epsilon/m + 1 - \epsilon & \text{if } a^* = \underset{a \in \mathcal{A}}{\operatorname{argmax}} \, Q(s,a) \\ \epsilon/m & \text{otherwise} \end{cases}
$$

---
+another way to make improvement faster is to improve the policy after each episode.

---
# GLIE ( Greedy in the Limit with Infinite Exploration )
it satisfies two conditions:
+ **Infinite Exploration** : Every action in every state continues to be explored infinitely often.This prevents the agent from missing potentially better actions early on.
+ **Greedy in the Limit** : As time goes on, the policy becomes increasingly greedy with respect to the learned value estimates.Eventually, the agent mostly chooses the action it currently believes is best.

it balances Exploration and Exploitation

Later in training:exploration gradually decreases,behavior becomes nearly greedy

+ To make it GLIE:ε must decrease over time ,but not too fast

+ Typical choice:
$$
ε_t = 1/t
$$

$ ϵ_t ⟶ 0 $ : eventually greedy

$ \sum_t ϵ_t = ∞ $ ⟶ infinite exploration

---
#SARSA
$$
Q(S,A) ⟵ Q(S,A) + α ( R + γ Q(S',A') - Q(S,A) )
$$
its algorithm:

Initialize $Q(s, a), \forall s \in \mathcal{S}, a \in \mathcal{A}(s)$, arbitrarily, and $Q(\text{terminal-state}, \cdot) = 0$

Repeat (for each episode):

&nbsp;&nbsp;&nbsp;&nbsp;Initialize $S$

&nbsp;&nbsp;&nbsp;&nbsp;Choose $A$ from $S$ using policy derived from $Q$ (e.g., $\varepsilon$-greedy)

&nbsp;&nbsp;&nbsp;&nbsp;Repeat (for each step of episode):

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Take action $A$, observe $R, S'$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Choose $A'$ from $S'$ using policy derived from $Q$ (e.g., $\varepsilon$-greedy)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$Q(S, A) \leftarrow Q(S, A) + \alpha [R + \gamma Q(S', A') - Q(S, A)]$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$S \leftarrow S'; A \leftarrow A';$

&nbsp;&nbsp;&nbsp;&nbsp;until $S$ is terminal

---
+ Sarsa converges to the optimal action-value function, under the following conditions:

GLIE sequence of policies , $\pi_t(a \mid s)$

Robbins-Monro sequence of step-sizes $α_t$
$$ \sum_{t=0}^{\infty} α_t = ∞ $$
$$  \sum_{k=0}^{\infty} α_t^2 < ∞ $$

---
#### Define the n-step Q-return

$$
q_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{n-1} R_{t+n} + \gamma^n Q(S_{t+n})
$$



#### Forward_view SARSA

Using weight $(1 - \lambda)\lambda^{n-1}$

$$
q_t^\lambda = (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} q_t^{(n)}
$$

$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left( q^λ_t - Q(S_t, A_t) \right)
$$

#### Backward_view SARSA

we use eligibility traces
$
\begin{aligned}
E_0(s, a) &= 0 \\
E_t(s, a) &= \gamma\lambda E_{t-1}(s, a) + \mathbf{1}(S_t = s, A_t = a)
\end{aligned}
$

$$
\begin{aligned}
\delta_t &= R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \\
Q(s, a) &\leftarrow Q(s, a) + \alpha\delta_t E_t(s, a)
\end{aligned}
$$

---
SARSA(λ) Algorithm

Initialize $Q(s, a)$ arbitrarily, for all $s \in \mathcal{S}, a \in \mathcal{A}(s)$

Repeat (for each episode):

&nbsp;&nbsp;&nbsp;&nbsp;$E(s, a) = 0$, for all $s \in \mathcal{S}, a \in \mathcal{A}(s)$

&nbsp;&nbsp;&nbsp;&nbsp;Initialize $S, A$

&nbsp;&nbsp;&nbsp;&nbsp;Repeat (for each step of episode):

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Take action $A$, observe $R, S'$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Choose $A'$ from $S'$ using policy derived from $Q$ (e.g., $\varepsilon$-greedy)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\delta \leftarrow R + \gamma Q(S', A') - Q(S, A)$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$E(S, A) \leftarrow E(S, A) + 1$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;For all $s \in \mathcal{S}, a \in \mathcal{A}(s)$:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$Q(s, a) \leftarrow Q(s, a) + \alpha \delta E(s, a)$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$E(s, a) \leftarrow \gamma \lambda E(s, a)$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$S \leftarrow S'; A \leftarrow A'$

&nbsp;&nbsp;&nbsp;&nbsp;until $S$ is terminal

---
## OFF POLICY
there is **TARGET POLICY** that we compute $v_{\pi}(s)$ or $q_{\pi}(s, a)$ and **BEHAVIOR POLICY** μ(a|s).

### OFF POLICY Monte-Carlo (importance sampling)
$$
G_t^{\pi/\mu} = \frac{\pi(A_t|S_t)}{\mu(A_t|S_t)} \frac{\pi(A_{t+1}|S_{t+1})}{\mu(A_{t+1}|S_{t+1})} \dots \frac{\pi(A_T|S_T)}{\mu(A_T|S_T)} G_t
$$

Update value towards *corrected* return

$$
V(S_t) \leftarrow V(S_t) + \alpha \left( G_t^{\pi/\mu} - V(S_t) \right)
$$
⇒ Cannot use if is μ zero when π is non-zero
it can dramatically increase variance (not recommended)

### OFF POLICY TD (importance sampling)
$$
\begin{aligned}
V(S_t) \leftarrow &V(S_t) + \alpha \left( \frac{\pi(A_t|S_t)}{\mu(A_t|S_t)} (R_{t+1} + \gamma V(S_{t+1})) - V(S_t) \right)
\end{aligned}
$$
Much lower variance than Monte-Carlo importance sampling
Policies only need to be similar over a single step

---
## Q-Learning

We now consider off-policy learning of action-values $Q(s, a)$ (No importance sampling is required)

+ Next action is chosen using behaviour policy $A_{t+1} \sim \mu(\cdot|S_t)$
+ But we consider alternative successor action $A' \sim \pi(\cdot|S_t)$
+ And update $Q(S_t, A_t)$ towards value of alternative action

$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left( R_{t+1} + \gamma Q(S_{t+1}, A') - Q(S_t, A_t) \right)
$$

+ The target policy $\pi$ improves greedy $Q(s, a)$

$$
\pi(S_{t+1}) = \underset{a'}{\operatorname{argmax}} \, Q(S_{t+1}, a')
$$

+ The behaviour policy $\mu$ improves for example with $\epsilon$-greedy $Q(s, a)$

### Q-Learning Algorithm

Initialize $Q(s, a), \forall s \in \mathcal{S}, a \in \mathcal{A}(s)$, arbitrarily, and $Q(\text{terminal-state}, \cdot) = 0$

Repeat (for each episode):

&nbsp;&nbsp;&nbsp;&nbsp;Initialize $S$

&nbsp;&nbsp;&nbsp;&nbsp;Repeat (for each step of episode):

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Choose $A$ from $S$ using policy derived from $Q$ (e.g., $\varepsilon$-greedy) (behavior policy)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Take action $A$, observe $R, S'$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma \max_{a} Q(S', a) - Q(S, A) \right]$ (target policy)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$S \leftarrow S';$

&nbsp;&nbsp;&nbsp;&nbsp;until $S$ is terminal

In [14]:
class PolicyImprovement:
    def __init__(self, num_states, num_actions):
        self.num_states = num_states
        self.num_actions = num_actions
        self.N_sa = np.zeros((num_states + 2, num_actions), dtype=int)

    def greedy_improvement(self, Q):
        best_actions = np.argmax(Q[1:-1], axis=1) # Exclude terminal states for policy
        return lambda state: int(best_actions[state - 1]) # Adjust for 1-indexed states

    def glie_epsilon_greedy_improvement(self, Q, episode_state_actions=None):
        if episode_state_actions is not None:
            for state, action in episode_state_actions:
                # Ensure state is within valid bounds before incrementing N_sa
                if 0 <= state < self.num_states + 2 and 0 <= action < self.num_actions:
                    self.N_sa[state, action] += 1

        def glie_policy(state):
            if state == 0 or state == self.num_states + 1: # Terminal states
                return 0 # Or any dummy action, as policy isn't defined here

            n_s = np.sum(self.N_sa[state])
            epsilon = 1.0 / n_s if n_s > 0 else 1.0

            if np.random.random() < epsilon:
                return int(np.random.randint(0, self.num_actions))
            else:
                return int(np.argmax(Q[state]))

        return glie_policy

    def display_policy(self, Q, randomwalk_env):
        """
        Displays the greedy policy derived from the Q-table for the RandomWalk environment.
        """
        symbols = {0: "←", 1: "→"} # Actions for RandomWalk

        print("\n===== LEARNED POLICY GRID =====")
        grid_str = []
        # Iterate through states, excluding the actual terminal states (0 and num_states+1)
        # which are part of the Q/V arrays for boundary conditions, but not for policy action.
        for s in range(1, randomwalk_env.num_states + 1):
            if s == randomwalk_env.left_terminal: # Leftmost actual state, not the terminal boundary
                grid_str.append("[G]") # Indicate goal/boundary
            elif s == randomwalk_env.right_terminal: # Rightmost actual state, not the terminal boundary
                grid_str.append("[G]") # Indicate goal/boundary
            else:
                # Get the best action for the current state (s)
                best_action = np.argmax(Q[s])
                grid_str.append(f" {symbols[best_action]} ")

        # Add explicit terminal states representation
        final_grid_display = []
        final_grid_display.append("[T]") # Left terminal
        final_grid_display.extend(grid_str)
        final_grid_display.append("[T]") # Right terminal

        print("".join(final_grid_display))
        print("=" * (randomwalk_env.num_states * 4 + 7)) # Adjust width based on num_states


In [15]:
EPISODES = 1000

mc_agent = MonteCarloPrediction(NUM_STATES, NUM_ACTIONS)
improver = PolicyImprovement(NUM_STATES, NUM_ACTIONS)

for _ in range(EPISODES):
    Q_table = mc_agent.mc_evaluation_q(randomwalk_env)

mc_agent.policy = improver.greedy_improvement(Q_table)

print("\n" + "=" * 35)
print("FINAL GREEDY POLICY")
print("=" * 35)
for s in range(1, NUM_STATES + 1):
    print(f"State {s} -> Target Action: {mc_agent.policy(s)}")
print("=" * 35)


FINAL GREEDY POLICY
State 1 -> Target Action: 1
State 2 -> Target Action: 1
State 3 -> Target Action: 1
State 4 -> Target Action: 1
State 5 -> Target Action: 1
State 6 -> Target Action: 1


In [16]:
EPISODES = 1000
ITERATIONS = 5

mc_agent = MonteCarloPrediction(NUM_STATES, NUM_ACTIONS)
improver = PolicyImprovement(NUM_STATES, NUM_ACTIONS)

for it in range(1, ITERATIONS + 1):
    all_episodes_history = []

    for _ in range(EPISODES):
        Q_table = mc_agent.mc_evaluation_q(randomwalk_env, num_episodes=1)

        raw_episode = mc_agent.run_episode(randomwalk_env)
        for step in raw_episode:
            all_episodes_history.append((step[0], step[1]))

    mc_agent.policy = improver.glie_epsilon_greedy_improvement(Q=Q_table,episode_state_actions=all_episodes_history)

print("=" * 50)
print(f"ITERATION {it}: GLIE Policy Matrix ({EPISODES} Episodes)")
print("=" * 50)
for s in range(1, NUM_STATES + 1):
    print(f"State {s} -> Target Action: {mc_agent.policy(s)}")
print("=" * 50 + "\n")

ITERATION 5: GLIE Policy Matrix (1000 Episodes)
State 1 -> Target Action: 1
State 2 -> Target Action: 1
State 3 -> Target Action: 1
State 4 -> Target Action: 1
State 5 -> Target Action: 1
State 6 -> Target Action: 1



In [17]:
EPISODES = 10000

mc_agent = MonteCarloPrediction(NUM_STATES, NUM_ACTIONS)
improver = PolicyImprovement(NUM_STATES, NUM_ACTIONS)

for episode in range(EPISODES):
    Q_table = mc_agent.mc_evaluation_q(randomwalk_env)

    raw_episode = mc_agent.run_episode(randomwalk_env)
    history = [(state, action) for state, action, reward in raw_episode]

    mc_agent.policy = improver.glie_epsilon_greedy_improvement( Q=Q_table, episode_state_actions=history)

print("\n" + "=" * 35)
print("FINAL LEARNED OPTIMAL POLICY")
print("=" * 35)
for s in range(1, NUM_STATES + 1):
    print(f"State {s} -> Target Action: {mc_agent.policy(s)}")
print("=" * 35)


FINAL LEARNED OPTIMAL POLICY
State 1 -> Target Action: 1
State 2 -> Target Action: 1
State 3 -> Target Action: 1
State 4 -> Target Action: 1
State 5 -> Target Action: 1
State 6 -> Target Action: 1


#SARSA

In [18]:
class SARSA:
    def __init__(self, num_states, num_actions, alpha=0.1, gamma=0.99, epsilon=0.1, lam=0.8):
        """Initializes the SARSA agent with configurations for all 3 variants."""
        self.num_states = num_states
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.lam = lam

        self.Q = np.zeros((num_states, num_actions))

    def _epsilon_greedy(self, state):
        """Helper method to choose an action using an epsilon-greedy policy."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.num_actions)

        # Break ties randomly if multiple actions have the max value
        max_actions = np.flatnonzero(self.Q[state] == self.Q[state].max())
        return np.random.choice(max_actions)

    def sarsa_0(self, env, num_episodes, max_steps_per_episode=2000):
        """1. SARSA(0): One-step Temporal Difference control."""
        for episode in range(num_episodes):
            state, _ = env.reset()
            action = self._epsilon_greedy(state)
            done = False
            steps = 0

            while not done and steps < max_steps_per_episode:
                next_state, reward, terminated, truncated, _ = env.step(action)

                steps += 1
                if steps >= max_steps_per_episode:
                    truncated = True

                done = terminated or truncated
                next_action = self._epsilon_greedy(next_state)

                # Calculate TD Target and TD Error
                td_target = reward + self.gamma * self.Q[next_state, next_action] * (1 - terminated)
                td_error = td_target - self.Q[state, action]

                # Online Q-Value update
                self.Q[state, action] += self.alpha * td_error

                state, action = next_state, next_action

    def sarsa_lambda_forward_view(self, env, num_episodes, max_steps_per_episode=1000):
        """2. SARSA(lambda) Forward View: Offline episodic updates using lambda-returns."""
        for episode in range(num_episodes):
            episode_history = []
            state, _ = env.reset()
            action = self._epsilon_greedy(state)
            done = False
            steps = 0

            #Collect full trajectory history up to safety limit
            while not done and steps < max_steps_per_episode:
                next_state, reward, terminated, truncated, _ = env.step(action)

                steps += 1
                if steps >= max_steps_per_episode:
                    truncated = True

                done = terminated or truncated
                next_action = self._epsilon_greedy(next_state)

                episode_history.append((state, action, reward, next_state, next_action))
                state, action = next_state, next_action

            # Update Q-values
            T = len(episode_history)
            for t in range(T):
                s_t, a_t, _, _, _ = episode_history[t]
                q_lambda_return = 0.0
                weight_sum = 0.0

                for n in range(1, T - t + 1):

                    n_step_return = sum((self.gamma ** i) * episode_history[t + i][2] for i in range(n))

                    # Apply bootstrapping checks
                    if t + n < T:
                        s_next, a_next = episode_history[t + n][0], episode_history[t + n][1]
                        n_step_return += (self.gamma ** n) * self.Q[s_next, a_next]
                    elif t + n == T:
                        final_state = episode_history[-1][3]
                        final_action = episode_history[-1][4]
                        if final_state != env.goal_state:
                            n_step_return += (self.gamma ** n) * self.Q[final_state, final_action]

                    weight = self.lam ** (n - 1) if (t + n == T) else (1 - self.lam) * (self.lam ** (n - 1))
                    q_lambda_return += weight * n_step_return
                    weight_sum += weight

                if weight_sum > 0:
                    q_lambda_return /= weight_sum

                self.Q[s_t, a_t] += self.alpha * (q_lambda_return - self.Q[s_t, a_t])

    def sarsa_lambda_backward_view(self, env, num_episodes, max_steps_per_episode=2000):
        """3. SARSA(lambda) Backward View: Online step updates utilizing eligibility traces."""
        for episode in range(num_episodes):

            E = np.zeros_like(self.Q)

            state, _ = env.reset()
            action = self._epsilon_greedy(state)
            done = False
            steps = 0

            while not done and steps < max_steps_per_episode:
                next_state, reward, terminated, truncated, _ = env.step(action)

                steps += 1
                if steps >= max_steps_per_episode:
                    truncated = True

                done = terminated or truncated
                next_action = self._epsilon_greedy(next_state)

                td_target = reward + self.gamma * self.Q[next_state, next_action] * (1 - terminated)
                td_error = td_target - self.Q[state, action]

                E[state, action] += 1

                self.Q += self.alpha * td_error * E
                E = self.gamma * self.lam * E

                state, action = next_state, next_action

In [19]:
class WindyGridworld:
    def __init__(self, height=7, width=10, start_state=(3, 0), goal_state=(3, 7),
                 wind_columns=None, wind_direction=(-1, 0), wind_enabled=True):
        """Highly optimized Gridworld Environment with configurable wind."""
        self.height = height
        self.width = width
        self.start_state = start_state
        self.goal_state = goal_state

        # Actions: 0: Up, 1: Down, 2: Left, 3: Right
        self.action_space = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.num_actions = len(self.action_space)
        self.num_states = height * width

        # Wind configurations
        self.wind_enabled = wind_enabled
        self.wind_direction = np.array(wind_direction)

        # Initialize wind settings using sets/dicts for O(1) lookups
        if wind_columns is None:
            self.wind_columns = {3, 4, 5, 6, 7, 8}
            self.wind_strengths = {3: 1, 4: 1, 5: 1, 6: 2, 7: 2, 8: 1}
        else:
            self.wind_columns = set(wind_columns)
            self.wind_strengths = {col: 1 for col in wind_columns}

        self.reset()

    def _coord_to_state(self, coord):
        """Converts (row, col) coordinate to a flat continuous state integer index."""
        return coord[0] * self.width + coord[1]

    def reset(self):
        """Resets the agent's position back to the start state."""
        self.agent_pos = np.array(self.start_state)
        return self._coord_to_state(self.agent_pos), {}

    def step(self, action_idx):
        """Executes a single step in the environment using vectorized operations."""
        # 1. Apply primary chosen action
        action_effect = np.array(self.action_space[action_idx])
        new_pos = self.agent_pos + action_effect

        # 2. Apply wind vector based on current column index
        curr_col = self.agent_pos[1]
        if self.wind_enabled and (curr_col in self.wind_columns):
            strength = self.wind_strengths.get(curr_col, 0)
            new_pos += self.wind_direction * strength

        # 3. Clip coordinates to keep the agent bounded within the grid matrix
        new_pos[0] = np.clip(new_pos[0], 0, self.height - 1)
        new_pos[1] = np.clip(new_pos[1], 0, self.width - 1)
        self.agent_pos = new_pos

        # 4. Compute reward structures and goal checks
        is_goal = np.array_equal(self.agent_pos, self.goal_state)
        reward = 0.0 if is_goal else -1.0

        return self._coord_to_state(self.agent_pos), reward, is_goal, False, {}

    def configure_wind(self, enabled=None, direction=None, columns=None, strengths=None):
        """Dynamically modifies environment wind attributes on the fly."""
        if enabled is not None:
            self.wind_enabled = enabled
        if direction is not None:
            self.wind_direction = np.array(direction)
        if columns is not None:
            self.wind_columns = set(columns)
        if strengths is not None:
            self.wind_strengths = strengths

In [20]:
def get_optimal_path(agent, env):
    """Traces the sequential trajectory from start to goal using the learned greedy policy."""
    path_coords = {}
    action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→'}

    state_idx, _ = env.reset()
    current_pos = tuple(env.agent_pos)
    done = False

    steps = 0
    max_steps = env.height * env.width

    while not done and steps < max_steps:

        best_action = np.argmax(agent.Q[state_idx])
        path_coords[current_pos] = action_symbols[best_action]

        state_idx, _, terminated, truncated, _ = env.step(best_action)
        current_pos = tuple(env.agent_pos)
        done = terminated or truncated
        steps += 1

    return path_coords

def display_isolated_path(agent, env):
    """Prints a clean, text-based grid visualizer displaying only the agent's path."""
    path_coords = get_optimal_path(agent, env)

    print(f"\n--- Optimal Path From Start State {env.start_state} ---")
    for r in range(env.height):
        row_symbols = []
        for c in range(env.width):
            if (r, c) == env.goal_state:
                row_symbols.append('★')  # Goal marker
            elif (r, c) in path_coords:
                row_symbols.append(path_coords[(r, c)])  # Directive action arrow
            else:
                row_symbols.append('.')  # Unvisited cell
        print(" ".join(row_symbols))
    print("-" * 50)

In [21]:
START_STATE = (0 , 0)
GOAL_STATE = (5 , 7)
HEIGHT = 7
WIDTH = 10

ALPHA = 0.5
GAMMA = 0.99
EPSILON = 0.1

windy_env = WindyGridworld(height=HEIGHT, width=WIDTH, start_state=START_STATE, goal_state=GOAL_STATE )

agent = SARSA(num_states=windy_env.num_states, num_actions=windy_env.num_actions, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON)

print("Training SARSA(0)...")
agent.sarsa_0(windy_env, num_episodes=2000)
print("Training finished successfully!")

display_isolated_path(agent, windy_env)

Training SARSA(0)...


Training finished successfully!

--- Optimal Path From Start State (0, 0) ---
→ → → → → → → → → ↓
. . . . . . . . . ↓
. . . . . . . . . ↓
. . . . . . . . . ↓
. . . . . . . . . ↓
. . . . . . . ★ . ↓
. . . . . . . . ← ←
--------------------------------------------------


In [22]:
START_STATE = (0 , 0)
GOAL_STATE = (5 , 7)
HEIGHT = 7
WIDTH = 10

ALPHA = 0.5
GAMMA = 0.99
EPSILON = 0.1

windy_env = WindyGridworld(height=HEIGHT, width=WIDTH, start_state=START_STATE, goal_state=GOAL_STATE )

agent = SARSA(num_states=windy_env.num_states, num_actions=windy_env.num_actions, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON)

print("Training SARSA(lambda) Backward View...")
agent.sarsa_lambda_backward_view(windy_env, num_episodes=500)
print("Training finished successfully!")

display_isolated_path(agent, windy_env)

Training SARSA(lambda) Backward View...


Training finished successfully!

--- Optimal Path From Start State (0, 0) ---
↓ . . . . . . . . .
↓ . . . . . . . . .
↓ . . . . . . . . .
→ ↓ . . . . . . . .
. ↓ → ↓ . . . . . .
. → → ← . . . ★ . .
. . . . . . . . . .
--------------------------------------------------


+ To run the SARSA($\lambda$) Forward View, we must handle it with extra care.Unlike SARSA(0) and the Backward View—which update the Q-table at every single step—the Forward View is an offline/episodic algorithm. It must collect a complete episode trace from start to finish before it can calculate the returns and update its values.Because it cannot learn during the episode, an untrained agent will wander randomly for a long time. Without a strict step limit, it will crash your memory or freeze your system.

#Q-LEARNING

In [23]:
class QLearning:
    def __init__(self, num_states, num_actions, alpha=0.1, gamma=0.99, epsilon=0.1):
        """Initializes the off-policy Q-Learning agent."""
        self.num_states = num_states
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

        self.Q = np.zeros((num_states, num_actions))

    def _epsilon_greedy(self, state):
        """Helper method to choose an action using an epsilon-greedy policy."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.num_actions)

        # Break ties randomly if multiple actions share the max value
        max_actions = np.flatnonzero(self.Q[state] == self.Q[state].max())
        return np.random.choice(max_actions)

    def train(self, env, num_episodes, max_steps_per_episode=2000):
        """Trains the agent using the off-policy Q-Learning algorithm."""
        for episode in range(num_episodes):
            state, _ = env.reset()
            done = False
            steps = 0

            while not done and steps < max_steps_per_episode:
                action = self._epsilon_greedy(state)
                next_state, reward, terminated, truncated, _ = env.step(action)

                steps += 1
                if steps >= max_steps_per_episode:
                    truncated = True

                done = terminated or truncated

                # Off-Policy update target: absolute maximum value of the next state
                max_next_q = np.max(self.Q[next_state])
                td_target = reward + self.gamma * max_next_q * (1 - terminated)
                td_error = td_target - self.Q[state, action]

                # Matrix value update step
                self.Q[state, action] += self.alpha * td_error
                state = next_state

In [24]:
START_STATE = (0 , 0)
GOAL_STATE = (5 , 7)
HEIGHT = 7
WIDTH = 10

ALPHA = 0.5
GAMMA = 0.99
EPSILON = 0.1

windy_env = WindyGridworld(height=HEIGHT, width=WIDTH, start_state=START_STATE, goal_state=GOAL_STATE )

ql_agent = QLearning(num_states=windy_env.num_states, num_actions=windy_env.num_actions, alpha=ALPHA, gamma=GAMMA , epsilon=EPSILON)

print("Training Q-Learning agent...")
ql_agent.train(windy_env, num_episodes=500)
print("Training complete!")

display_isolated_path(ql_agent, windy_env)

Training Q-Learning agent...


Training complete!

--- Optimal Path From Start State (0, 0) ---
→ → → → → → → → → ↓
. . . . . . . . . ↓
. . . . . . . . . ↓
. . . . . . . . . ↓
. . . . . . . . . ↓
. . . . . . . ★ . ↓
. . . . . . . . ← ←
--------------------------------------------------


The Key Difference (SARSA vs Q-Learning)SARSA is On-Policy. Its update equation relies on $Q(S_{t+1}, A_{t+1})$, meaning it takes into account that it might make a random exploratory move next. In dangerous environments (like Cliff Walking), SARSA learns a safer, more conservative path.Q-Learning is Off-Policy. Its update equation relies on $\max_a Q(S_{t+1}, a)$, assuming it will always take the optimal action in the future even if it's currently exploring randomly. This allows it to learn the true mathematical shortest path much faster.

In [25]:
class CliffWalking:
    def __init__(self, height=4, width=12, start_state=(3, 0), goal_state=(3, 11)):
        """Highly optimized Cliff Walking Environment compatible with RL agents."""
        self.height = height
        self.width = width
        self.start_state = start_state
        self.goal_state = goal_state

        # Actions: 0: Up, 1: Down, 2: Left, 3: Right
        self.action_space = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.num_actions = len(self.action_space)
        self.num_states = height * width

        # Dynamic cliff logic based on the bottom row between start and goal columns
        self.cliff_row = self.height - 1
        self.cliff_columns = set(range(min(start_state[1], goal_state[1]) + 1,
                                       max(start_state[1], goal_state[1])))

        self.reset()

    def _coord_to_state(self, coord):
        """Converts (row, col) coordinate to a flat continuous state integer index."""
        return coord[0] * self.width + coord[1]

    def reset(self):
        """Resets the agent's position back to the start state."""
        self.agent_pos = np.array(self.start_state)
        return self._coord_to_state(self.agent_pos), {}

    def step(self, action_idx):
        """Executes a single step in the cliff walking environment."""
        # Apply primary chosen action
        action_effect = np.array(self.action_space[action_idx])
        new_pos = self.agent_pos + action_effect

        # Clip coordinates to keep the agent bounded within the grid matrix
        new_pos[0] = np.clip(new_pos[0], 0, self.height - 1)
        new_pos[1] = np.clip(new_pos[1], 0, self.width - 1)

        # cliff collision
        is_cliff = (new_pos[0] == self.cliff_row) and (new_pos[1] in self.cliff_columns)

        if is_cliff:
            # High penalty and immediate position reset
            reward = -100.0
            self.agent_pos = np.array(self.start_state)
            terminated = False  # Episode continues from start state per Sutton & Barto definition
        else:
            self.agent_pos = new_pos
            is_goal = np.array_equal(self.agent_pos, self.goal_state)
            reward = 0.0 if is_goal else -1.0
            terminated = is_goal

        return self._coord_to_state(self.agent_pos), reward, terminated, False, {}

In [26]:
START_STATE = (3, 0)
GOAL_STATE = (3, 11)
HEIGHT = 4
WIDTH = 12

ALPHA = 0.5
GAMMA = 0.99
EPSILON = 0.1

EPISODE_NUM = 500

cliff_env = CliffWalking(height=HEIGHT, width=WIDTH, start_state=START_STATE, goal_state=GOAL_STATE)



# Train SARSA Agent (On-Policy)
sarsa_agent = SARSA(num_states=cliff_env.num_states, num_actions=cliff_env.num_actions,
                    alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON)

sarsa_agent.sarsa_0(cliff_env, num_episodes=EPISODE_NUM)

print("\n=== SARSA Path (Safer, stays away from the cliff edge) ===")
display_isolated_path(sarsa_agent, cliff_env)



# Train Q-Learning Agent (Off-Policy)
ql_agent = QLearning(num_states=cliff_env.num_states, num_actions=cliff_env.num_actions,
                     alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON)

ql_agent.train(cliff_env, num_episodes=500)

print("\n=== Q-Learning Path (Optimal, walks directly along the cliff edge) ===")
display_isolated_path(ql_agent, cliff_env)


=== SARSA Path (Safer, stays away from the cliff edge) ===

--- Optimal Path From Start State (3, 0) ---
→ → → → → → → → → → → ↓
↑ . . . . . . . . . . ↓
↑ . . . . . . . . . . ↓
↑ . . . . . . . . . . ★
--------------------------------------------------



=== Q-Learning Path (Optimal, walks directly along the cliff edge) ===

--- Optimal Path From Start State (3, 0) ---
. . . . . . . . . . . .
. . . . . . . . . . . .
→ → → → → → → → → → → ↓
↑ . . . . . . . . . . ★
--------------------------------------------------
